# Train fresh Rythu Mitra models in Google Colab

This notebook trains from scratch using newly downloaded/uploaded datasets. It does **not** load or fine-tune the model files already in the repository. Run with a Colab GPU (Runtime → Change runtime type → T4 GPU). The plant disease dataset downloads automatically. For crop recommendations, download the licensed [Mendeley seven-feature crop dataset](https://data.mendeley.com/datasets/2mht7hz98y/1) and upload its CSV when prompted. Its page lists N, P, K, temperature, humidity, pH, and rainfall, matching the backend's input features.

The disease model is trained on PlantVillage images (controlled backgrounds); its scores are dataset validation scores, not proof of field accuracy. Both models are preliminary decision support. The notebook matches common crop CSV column names automatically; no synthetic or old repository training data is used. At the end, download `rythu_mitra_fresh_models.zip` and place its files in the repository paths listed in the final cell.

In [ ]:
!pip -q install tensorflow-datasets pandas scikit-learn
import json, os, hashlib, zipfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_datasets as tfds
from google.colab import files
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Train a new disease classifier

Downloads PlantVillage through TensorFlow Datasets (54,303 labeled images, 38 classes). A seeded 80/20 split is used; the model is initialized from scratch. The exported class list is taken directly from the dataset metadata and stored in the same order as the model outputs.

In [ ]:
DATA_DIR = '/content/tfds'
SIZE = 128
BATCH = 64
EPOCHS = 12
builder = tfds.builder('plant_village', data_dir=DATA_DIR)
builder.download_and_prepare()
info = builder.info
class_names = list(info.features['label'].names)
total = info.splits['train'].num_examples
assert len(class_names) == 38 and total > 50000, (len(class_names), total)
raw = builder.as_dataset(split='train', as_supervised=True)
raw = raw.shuffle(total, seed=42, reshuffle_each_iteration=False)
val_count = int(total * 0.2)
val_raw, train_raw = raw.take(val_count), raw.skip(val_count)
def prep(image, label):
    image = tf.image.resize(image, (SIZE, SIZE), antialias=True)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_raw.map(prep, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)
val_ds = val_raw.map(prep, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)
model = tf.keras.Sequential([
    tf.keras.Input(shape=(SIZE, SIZE, 3)),
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'), tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'), tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'), tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(192, 3, padding='same', activation='relu'), tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.35), tf.keras.layers.Dense(len(class_names), activation='softmax')
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
_, disease_val_accuracy = model.evaluate(val_ds, verbose=0)
OUT = Path('/content/rythu_mitra_models')
OUT.mkdir(exist_ok=True)
model.save(OUT / 'plant_disease_model.h5')
(OUT / 'disease_labels.json').write_text(json.dumps(class_names, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT / 'disease_training_metrics.json').write_text(json.dumps({'dataset':'TensorFlow Datasets PlantVillage','dataset_version':str(info.version),'dataset_examples':total,'classes':len(class_names),'split':'seeded 80/20 image split','epochs_run':len(history.history['loss']),'validation_accuracy':float(disease_val_accuracy),'input_size':[SIZE,SIZE,3],'preprocessing':'RGB resized to 128x128 and scaled to [0,1]','trained_from_scratch':True}, indent=2), encoding='utf-8')
print(f'Disease model saved; validation accuracy: {disease_val_accuracy:.3f}; classes: {len(class_names)}')

## 2. Train a new crop recommender from the licensed CSV

Upload the CSV downloaded from the Mendeley source linked above. The notebook prints the CSV headers and automatically matches common names such as `Nitrogen`/`N`, `Phosphorus`/`P`, `Temperature`/`Temp`, `Humidity`, `Rainfall`, `pH`, and `Crop`/`Label`. You do not need to edit code for these. It stops with an error if a required real column is missing; it never makes up values.

In [ ]:
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
assert len(csv_files) == 1, 'Upload exactly one CSV file.'
crop_df = pd.read_csv(csv_files[0])
print('CSV columns:', crop_df.columns.tolist())
display(crop_df.head())
FEATURES = ['N','P','K','temperature','humidity','ph','rainfall']
# The notebook recognizes common spellings automatically; no edits should be needed.
def norm_name(value):
    return ''.join(ch for ch in str(value).lower() if ch.isalnum())
ALIASES = {
    'N':['n','nitrogen'], 'P':['p','phosphorus','phosphorous'], 'K':['k','potassium'],
    'temperature':['temperature','temp','avgtemperature','meantemperature'],
    'humidity':['humidity','relativehumidity'], 'ph':['ph','soilph'],
    'rainfall':['rainfall','rain'], 'label':['label','crop','crops','cropname','recommendedcrop']
}
COLUMN_MAP = {}
for field, aliases in ALIASES.items():
    matches = [column for column in crop_df.columns if norm_name(column) in aliases]
    if matches:
        COLUMN_MAP[field] = matches[0]
missing = [field for field in FEATURES + ['label'] if field not in COLUMN_MAP]
assert not missing, f'Could not identify these required columns: {missing}. Printed CSV columns: {crop_df.columns.tolist()}'
print('Automatically matched CSV columns:', COLUMN_MAP)
crop_data = crop_df[[COLUMN_MAP[k] for k in FEATURES + ['label']]].copy()
crop_data.columns = FEATURES + ['label']
for column in FEATURES: crop_data[column] = pd.to_numeric(crop_data[column], errors='coerce')
crop_data = crop_data.dropna(subset=FEATURES + ['label'])
crop_data['label'] = crop_data['label'].astype(str).str.strip()
crop_data = crop_data[crop_data['label'] != '']
assert len(crop_data) >= 100, f'Only {len(crop_data)} clean rows remain; check the selected source columns.'
assert crop_data['label'].nunique() >= 2, 'At least two crop labels are required.'
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val = train_test_split(crop_data[FEATURES], crop_data['label'], test_size=0.2, random_state=42, stratify=crop_data['label'])
crop_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced_subsample')
crop_model.fit(x_train, y_train)
crop_val_accuracy = accuracy_score(y_val, crop_model.predict(x_val))
import pickle
with (OUT / 'crop_model.pkl').open('wb') as f: pickle.dump(crop_model, f)
crop_hash = hashlib.sha256(Path(csv_files[0]).read_bytes()).hexdigest()
(OUT / 'crop_training_metrics.json').write_text(json.dumps({'dataset_file':csv_files[0],'dataset_sha256':crop_hash,'rows_used':len(crop_data),'classes':sorted(crop_data['label'].unique().tolist()),'feature_order':FEATURES,'split':'stratified 80/20 random_state=42','validation_accuracy':float(crop_val_accuracy),'classification_report':classification_report(y_val, crop_model.predict(x_val), output_dict=True, zero_division=0),'trained_from_scratch':True}, indent=2), encoding='utf-8')
print(f'Crop model saved; validation accuracy: {crop_val_accuracy:.3f}; rows: {len(crop_data)}')

## 3. Package and download

The zip contains the disease model and exact dataset-derived class order at the paths expected by the backend, plus crop model and validation/provenance reports. Training metrics are not field-performance claims. Review dataset rights and results before deployment.

In [ ]:
zip_path = '/content/rythu_mitra_fresh_models.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for name in ['plant_disease_model.h5','disease_labels.json','disease_training_metrics.json','crop_model.pkl','crop_training_metrics.json']:
        path = OUT / name
        if path.exists(): z.write(path, arcname=name)
print('Included files:', sorted(p.name for p in OUT.iterdir()))
print('Backend install paths: crop_model.pkl → models/crop_model.pkl; plant_disease_model.h5 → models/plant_disease_model.h5; disease_labels.json → data/disease_labels.json')
files.download(zip_path)